# ETL

## 0. Importación y preparación del entorno

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Extracción

In [2]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Transformación

In [3]:
# Limpiamos TotalCharges: convertimos a numérico y reemplazamos espacios vacíos por
# `NaN`.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [4]:
# Eliminamos customerID y separamos la variable objetivo.
df = df.drop(columns=['customerID'])

X = df.drop(columns=['Churn'])
y = df['Churn']

In [5]:
# Dividimos los datos y simulamos test_size=0.2 y random_state=42.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# Calculamos la mediana de TotalCharges con el set de entrenamiento.
median_train = X_train['TotalCharges'].median()

# Rellenamos los huecos en ambos sets con esa misma mediana y así evitamos
# fuga de datos.
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(median_train)
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(median_train)

In [7]:
# Codificamos a 0/1:

# gender y partner
mapping_gender = {'Female': 1, 'Male': 0}
mapping_yes_no = {'Yes': 1, 'No': 0}

X_train['gender'] = X_train['gender'].map(mapping_gender)
X_test['gender'] = X_test['gender'].map(mapping_gender)

X_train['Partner'] = X_train['Partner'].map(mapping_yes_no)
X_test['Partner'] = X_test['Partner'].map(mapping_yes_no)

# churn
y_train = y_train.map(mapping_yes_no)
y_test = y_test.map(mapping_yes_no)

## 3. Carga

In [8]:
# Definir la ruta de la carpeta de destino
processed_data_dir = "../data/processed"

# Creamos la carpeta si aún no existe (para evitar errores)
os.makedirs(processed_data_dir, exist_ok=True)

# Guardamos cada conjunto como un archivo CSV independiente
X_train.to_csv(os.path.join(processed_data_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(processed_data_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(processed_data_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(processed_data_dir, "y_test.csv"), index=False)

print(f"Los 4 datasets procesados han sido guardados en '{processed_data_dir}/'")

Los 4 datasets procesados han sido guardados en '../data/processed/'
